Neural Network

Generated by MLguide. This notebook is a starter workflow for **classification** using a **PyTorch neural network**.

# Problem context
The NASA Turbofan Engine Degradation Simulation Dataset (C-MAPSS) was developed by NASA to support research in engine health monitoring and prognostics. It contains multivariate time-series data from multiple simulated turbofan engines operated under different conditions and fault modes. Each engine is simulated from an initial healthy state until failure, allowing the observation of progressive performance degradation. For every engine cycle, the dataset provides three operational settings and 21 sensor measurements. The simulated environment enables controlled variation of operating regimes and fault scenarios, making the dataset suitable for analyzing system behavior over time. The full dataset and description can be accessed at \url{https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data}
 (last accessed: \today).

# 1. Environment setup

In [ ]:
%pip install -q pandas numpy scikit-learn torch

# 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

# 3. Load and prepare data

In [ ]:
data_path = "data.csv"
target_column = "target"

df = pd.read_csv(data_path)
X_raw = df.drop(columns=[target_column])
y_raw = df[target_column]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
num_classes = len(label_encoder.classes_)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_cols = X_train_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_train_raw.columns if c not in numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
    ]
)

X_train = preprocess.fit_transform(X_train_raw)
X_test = preprocess.transform(X_test_raw)

if hasattr(X_train, "toarray"):
    X_train = X_train.toarray()
if hasattr(X_test, "toarray"):
    X_test = X_test.toarray()

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

# 4. Define model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nn.Sequential(
    nn.Linear(X_train_t.shape[1], 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, num_classes),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 5. Train and evaluate

In [ ]:
num_epochs = 20
model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{num_epochs} - loss: {epoch_loss:.4f}")

model.eval()
with torch.no_grad():
    logits = model(X_test_t.to(device))
    pred = torch.argmax(logits, dim=1).cpu().numpy()

acc = accuracy_score(y_test, pred)
print(f"Accuracy: {acc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, pred, target_names=label_encoder.classes_.astype(str)))

# 6. Method notes
- Use GPU (`cuda`) for larger datasets and deeper networks.
- Tune layer widths, dropout, learning rate, and epochs for your task.